<a href="https://colab.research.google.com/github/TheAlishbahWaheed/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TheAlishbahWaheed/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*


A content team managing a few thousand live pages cannot review all of them every
week. Someone has to pick which pages get a second look first — and today that pick
is usually a hand-written rule ("anything untouched for 90 days") or a gut call.

The cost of getting the pick wrong runs in both directions: spend a reviewer's
morning on a page that was never actually declining, and a genuinely slipping page
sits untouched for another month.

**Research question:** using only observed, page-level search and content signals,
can a model rank pages by decline risk better than a transparent staleness rule can
— and by how much, once the comparison is made honest?

**The decision this supports:** which of a few thousand live FlyRank pages a content
editor should prioritize for review this week. The deliverable is not a black box —
it's a ranked queue with a reason code attached to every row, so an editor can see
*why* a page was flagged, agree or disagree with the model in seconds, and act (or
not) on their own judgment.

In [10]:
pass

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*


**Release:** `data/raw/content_refresh_anonymized.csv` — the 30,000-page, 32-client
anonymized FlyRank starter slice. One row = one content page (`content_id`),
pseudonymized under one of 32 clients (`client_id`). Confirmed below: 30,000 rows,
30,000 unique `content_id`, zero duplicates.

**Time window:** performance metrics (impressions, clicks, sessions, `ctr`,
`avg_position`, etc.) are aggregated over a trailing 90-day window ending at export,
with `*_last_30d` / `*_prev_30d` sub-windows for trend comparison. Two lifecycle
fields run on a separate, longer clock: `content_age_days` (90–564 days) and
`days_since_last_update` (1–373 days). This is a single snapshot — no repeated
observations of the same page over time.

**Excluded, and why:**
- Client names, domains, URLs, page titles, raw queries — never present in the
  release; only hashed `content_id` / `client_id` remain, used for grouping only.
- `trend_direction` / `trend_pct` — these *define* the label, so they can never also
  be model inputs.
- `impressions_last_30d`/`_prev_30d` and click/session equivalents — used to build
  the label, not to predict it.
- FlyRank's own product decision flags (`health_score`, `needs_ctr_fix`,
  `is_quick_win`) — intentionally left out so the model learns from observable
  evidence, not a copy of an existing product decision.
- `provider_used` (71.5% missing) and `model_used` (19.1% missing) — internal
  generation-pipeline metadata, not a page signal.

**Missingness is not random:** `word_count`/`char_count` are complete for
`comparison article` and `feedly article` rows but ~28% missing for
`keyword article` — 90.7% of the dataset by row count. A blind `fillna(0)` would
silently tell the model "this page has zero words," so an explicit
`has_word_count` flag carries the missingness forward instead of erasing it.

In [1]:
import os
import pandas as pd

# Clone the repo if this is a fresh Colab session and the data isn't local yet
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.exists("flyrank-ml-internship"):
        !git clone https://github.com/TheAlishbahWaheed/flyrank-ml-internship.git
    os.chdir("flyrank-ml-internship")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Shape:", df.shape)
print("Unique content_id:", df["content_id"].nunique(), "| duplicated rows:", df["content_id"].duplicated().sum())
print("Unique clients:", df["client_id"].nunique())

# Windows
print("\ncontent_age_days range:", df["content_age_days"].min(), "-", df["content_age_days"].max())
print("days_since_last_update range:", df["days_since_last_update"].min(), "-", df["days_since_last_update"].max())

# Rows per client — not a balanced panel
per_client = df.groupby("client_id").size()
print("\nRows per client — min/median/max:", per_client.min(), per_client.median(), per_client.max())

# Missingness overall, and where it's patterned
print("\nTop missing columns (%):")
print((df.isna().mean() * 100).round(1).sort_values(ascending=False).head(8))

print("\nword_count missing % by content_type:")
print((df.groupby("content_type")["word_count"].apply(lambda s: s.isna().mean() * 100)).round(1))

# The label itself — built here for reference only, never used as a feature elsewhere
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)
print("\nBase decline rate (whole slice):", round(df["is_declining_label"].mean(), 3))

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 269, done.
remote: Counting objects: 100% (269/269), done.
remote: Compressing objects: 100% (223/223), done.
remote: Total 269 (delta 122), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (269/269), 2.08 MiB | 12.66 MiB/s, done.
Resolving deltas: 100% (122/122), done.
Shape: (30000, 44)
Unique content_id: 30000 | duplicated rows: 0
Unique clients: 32

content_age_days range: 90 - 564
days_since_last_update range: 1 - 373

Rows per client — min/median/max: 3 567.0 7008

Top missing columns (%):
provider_used        71.5
word_count           25.7
char_count           25.7
word_count_tier      25.7
char_count_tier      25.7
model_used           19.1
trend_pct            11.3
competition_level     8.7
dtype: float64

word_count missing % by content_type:
content_type
comparison article     0.0
feedly article         0.0
keyword article       28.3
Name: word_count, dtype: float64

Base decline rate (w

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*



**Framing:** ranking problem — *which pages first?* — scored with precision@K. Target
`is_declining_label` = 1 when `trend_direction == "down"` (base rate 0.542 across all
30,000 rows) — an observed label, not an invented rule.

**Baseline:** transparent rule, built first and never touching the label — flag a page
in the 91–180 day staleness tier AND the moderate/good impression tier, score by
`impressions_90d`, rank descending. Same "stale + visible" logic behind FlyRank's own
refresh flags.

**Features (29 total):** demand (`search_volume`, `competition`, `cpc`), content
(`word_count`, `char_count`, log-transformed 90-day traffic counts), lifecycle
(`content_age_days`, `days_since_last_update`), current-state rates (`ctr`,
`avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`), and 3 `has_*`
missingness flags. Explicitly excluded: `trend_direction`, `trend_pct`, every
`*_last_30d`/`*_prev_30d` column, plus ID columns.

**Models:** Logistic Regression (readable, coefficient-per-feature start) and Random
Forest (`max_depth=8`, `min_samples_leaf=20` — deliberately shallow, since 30k rows
across only 32 clients invites memorizing client quirks rather than learning a
transferable signal).

**Validation design:** split by `client_id` with `GroupShuffleSplit` (75/25,
`random_state=42`) — every client's pages land entirely in train or entirely in test,
zero overlap. To make that choice honest rather than assumed, the identical pipeline
was re-run once under a naive random split and once under the grouped split: ROC-AUC
dropped from 0.752 (naive, client leakage) to 0.603 (grouped, zero overlap) — a 0.149
point drop purely from closing the leak. Every result reported below uses the honest,
grouped number.

**Leakage checks:**
1. The 90-day feature windows structurally contain the 30-day windows the label is
   built from (100% of rows satisfy `last_30d + prev_30d ≤ 90d`; correlation 0.918) —
   a soft, disclosed risk, not fatal: dropping every `*_90d` feature changed grouped
   AUC by only −0.008 (0.603 → 0.595).
2. Sanity check: deliberately adding `trend_pct` and the last/prev-30d windows back
   into the feature set pushed AUC to a suspicious 1.000, confirming the
   leakage-detection method works and that 0.603 isn't hiding a similar leak.
3. No column names in the release match FlyRank's own product-flag naming pattern.

In [2]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

# Leakage-safe feature set
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_scroll_data"] = df["scroll_rate"].notna().astype(int)

NUM_FEATS = ["search_volume","competition","cpc","word_count","char_count",
    "log_impressions_90d","log_clicks_90d","log_sessions_90d","log_ai_sessions_90d",
    "days_with_impressions","days_with_sessions","content_age_days","days_since_last_update",
    "ctr","avg_position","engagement_rate","scroll_rate","ai_traffic_pct",
    "has_keyword_data","has_word_count","has_scroll_data"]
CAT_FEATS = ["competition_level","content_type","main_intent","age_tier","freshness_tier",
    "word_count_tier","impression_tier","position_tier"]

for c in ["search_volume","competition","cpc","word_count","char_count","scroll_rate"]:
    df[c] = df[c].fillna(0)
for c in CAT_FEATS:
    df[c] = df[c].fillna("unknown").astype(str)

X = df[NUM_FEATS + CAT_FEATS]
y = df["is_declining_label"].values
groups = df["client_id"].values

# Honest, client-grouped split — zero client overlap
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(df, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
print("Client overlap between train/test (must be 0):",
      len(set(df.iloc[train_idx]["client_id"]) & set(df.iloc[test_idx]["client_id"])))

# Baseline
stale = (df["freshness_tier"] == "91-180").astype(int)
visible = df["impression_tier"].isin(["moderate", "good"]).astype(int)
df["baseline_score"] = stale * visible * df["impressions_90d"]
baseline_test_scores = df.iloc[test_idx]["baseline_score"].values

# Logistic Regression
lr_pre = ColumnTransformer([("num", StandardScaler(), NUM_FEATS), ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATS)])
logreg = Pipeline([("pre", lr_pre), ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])
logreg.fit(X_train, y_train)
logreg_proba = logreg.predict_proba(X_test)[:, 1]

# Random Forest
rf_pre = ColumnTransformer([("num", "passthrough", NUM_FEATS), ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATS)])
rf = Pipeline([("pre", rf_pre), ("clf", RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=RANDOM_STATE, n_jobs=-1))])
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(y_true)[order[:k]].mean())

print("ROC-AUC — baseline:", round(roc_auc_score(y_test, baseline_test_scores), 3),
      "| logreg:", round(roc_auc_score(y_test, logreg_proba), 3),
      "| rf:", round(roc_auc_score(y_test, rf_proba), 3))

Client overlap between train/test (must be 0): 0
ROC-AUC — baseline: 0.492 | logreg: 0.61 | rf: 0.603


In [3]:
from sklearn.model_selection import train_test_split

def fit_and_score(train_idx, test_idx, label):
    pre = ColumnTransformer([("num", "passthrough", NUM_FEATS), ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATS)])
    rf2 = Pipeline([("pre", pre), ("clf", RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=RANDOM_STATE, n_jobs=-1))])
    rf2.fit(X.iloc[train_idx], y[train_idx])
    proba = rf2.predict_proba(X.iloc[test_idx])[:, 1]
    return {"split": label, "roc_auc": round(roc_auc_score(y[test_idx], proba), 3)}

# BEFORE: naive random split (client leakage possible)
tr_r, te_r = train_test_split(np.arange(len(df)), test_size=0.25, random_state=RANDOM_STATE, stratify=y)
row_random = fit_and_score(tr_r, te_r, "naive (random)")

# AFTER: same pipeline, client-grouped (honest)
row_grouped = fit_and_score(train_idx, test_idx, "grouped (honest)")
print(pd.DataFrame([row_random, row_grouped]).to_string(index=False))
print(f"ROC-AUC drop from closing the leak: {row_random['roc_auc'] - row_grouped['roc_auc']:+.3f}")

# Leakage check 1: window overlap
wc = df[["impressions_90d","impressions_last_30d","impressions_prev_30d"]].dropna().copy()
wc["sum_60d"] = wc["impressions_last_30d"] + wc["impressions_prev_30d"]
print("\nFraction rows where last_30d+prev_30d <= 90d:", round((wc["sum_60d"] <= wc["impressions_90d"]+1e-6).mean(), 3))
print("Correlation(impressions_90d, impressions_last_30d):", round(df["impressions_90d"].corr(df["impressions_last_30d"]), 3))

# Leakage check 2: cost of dropping *_90d features
NUM_NO_90D = [f for f in NUM_FEATS if "_90d" not in f]
def score_custom(fn, fc):
    pre = ColumnTransformer([("num","passthrough",fn),("cat",OneHotEncoder(handle_unknown="ignore"),fc)])
    m = Pipeline([("pre",pre),("clf",RandomForestClassifier(n_estimators=300,max_depth=8,min_samples_leaf=20,random_state=RANDOM_STATE,n_jobs=-1))])
    m.fit(X[fn+fc].iloc[train_idx], y[train_idx])
    p = m.predict_proba(X[fn+fc].iloc[test_idx])[:,1]
    return roc_auc_score(y[test_idx], p)
print("\nFull feature set AUC:", round(score_custom(NUM_FEATS, CAT_FEATS), 3))
print("Minus *_90d features AUC:", round(score_custom(NUM_NO_90D, CAT_FEATS), 3))

# Leakage check 3: no product-flag columns present
flags = [c for c in df.columns if "flag" in c.lower() or "health" in c.lower() or "quick_win" in c.lower()]
print("\nProduct-flag-pattern columns found:", flags or "none")

           split  roc_auc
  naive (random)    0.752
grouped (honest)    0.603
ROC-AUC drop from closing the leak: +0.149

Fraction rows where last_30d+prev_30d <= 90d: 1.0
Correlation(impressions_90d, impressions_last_30d): 0.918

Full feature set AUC: 0.603
Minus *_90d features AUC: 0.595

Product-flag-pattern columns found: none


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*



Both models and the baseline were scored on the identical grouped test split —
7,115 pages from 8 clients never seen in training (test base rate 0.517).

| K   | Baseline (rule) | Logistic Regression | Random Forest |
|-----|------------------|----------------------|----------------|
| 10  | 0.60             | 0.90                 | 0.40           |
| 20  | 0.45             | 0.80                 | 0.55           |
| 50  | 0.38             | 0.74                 | 0.54           |
| 100 | 0.32             | 0.71                 | 0.56           |
| 200 | 0.44             | 0.67                 | 0.565          |
| **ROC-AUC** | 0.492   | 0.61                 | 0.603          |

Logistic Regression edged out Random Forest at every rank depth on this split — a
useful, slightly humbling result (a simple, readable model outperformed a more
complex one here). Random Forest was still carried forward as the production scorer:
the Week-4 signal audit found non-monotonic staleness/impression-tier patterns a
linear model can't represent, and a single grouped split is one draw from a
32-client population small enough that model rankings can flip on the next draw.

To reduce single-split noise, the deployed action queue is scored with 5-fold
client-grouped out-of-fold (OOF) Random Forest predictions instead: cross-validated
AUC **0.673**, fold range **0.602–0.735** — the spread itself is a finding (expect
swings this size with only 32 clients, not model instability).

![Precision@K](https://github.com/TheAlishbahWaheed/flyrank-ml-internship/blob/main/work/figures/precision_at_k.png?raw=1)
*Ranked-queue precision@K vs. base rate — precision climbs from 0.40 at K=50 to 0.71
at K=1000 as the queue widens. Most of this model's value shows up in batches of
hundreds, not a handful of top picks.*

In [4]:
def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(y_true)[order[:k]].mean())

base_rate = y_test.mean()
rows = []
for k in [10, 20, 50, 100, 200]:
    rows.append({
        "k": k,
        "base_rate": round(base_rate, 3),
        "baseline_precision@k": round(precision_at_k(y_test, baseline_test_scores, k), 3),
        "logreg_precision@k": round(precision_at_k(y_test, logreg_proba, k), 3),
        "rf_precision@k": round(precision_at_k(y_test, rf_proba, k), 3),
    })
print(pd.DataFrame(rows).to_string(index=False))
print("\nROC-AUC — baseline:", round(roc_auc_score(y_test, baseline_test_scores), 3),
      "| logreg:", round(roc_auc_score(y_test, logreg_proba), 3),
      "| rf:", round(roc_auc_score(y_test, rf_proba), 3))

  k  base_rate  baseline_precision@k  logreg_precision@k  rf_precision@k
 10      0.517                  0.60                0.90           0.400
 20      0.517                  0.45                0.80           0.550
 50      0.517                  0.38                0.74           0.540
100      0.517                  0.32                0.71           0.560
200      0.517                  0.44                0.67           0.565

ROC-AUC — baseline: 0.492 | logreg: 0.61 | rf: 0.603


## 5. Limitations

*What this work cannot claim.*


**Modest, not strong, discrimination.** Grouped ROC-AUC of 0.603 is better than
chance (0.5) but well short of a strong classifier; precision@50 (0.540) sits barely
above that split's own base rate (0.517). Most of this model's value shows up across
hundreds of ranked rows, not the very top picks.

**Cross-client generalization is the real weak point, not model choice.** The
naive-vs-grouped check (Section 3) found ROC-AUC fell from 0.752 to 0.603 the moment
client leakage was removed — the honest number is meaningfully weaker than a
careless one, and any future re-run should keep reporting both.

**One content type dominates.** `keyword article` is 90.7% of rows; findings may not
transfer cleanly to the other two content types, which are too thin here to evaluate
separately with confidence.

**Single 90-day snapshot, not a trajectory.** One row per page means the model sees
the current 90 days, not how a page's metrics moved over its life — the label is a
trend proxy, not a verified outcome tracked forward.

**Not the full production warehouse.** This paper used the 30,000-row anonymized
starter slice, not the ~79M-row FlyRank/internship-warehouse release — findings
describe this slice and should be treated as a pilot, not a claim about FlyRank's
full content base.

**No causal claims.** Every finding here is an observed association in
cross-sectional data. Nothing in this paper shows that refreshing a flagged page
causes recovery, and nothing here models or predicts a search engine's ranking
algorithm.

In [9]:
# Numbers backing the limitations above
print("content_type share (limit: one type dominates):")
print(df["content_type"].value_counts(normalize=True).round(3))

print("\nGrouped-split AUC (audited, from Section 3):", round(auc_single, 3))
print("Naive-split AUC (from Section 3's honesty check):", round(row_random["roc_auc"], 3))
print("Precision@50 vs. base rate:", round(precision_at_k(y_test, rf_proba, 50), 3), "vs.", round(y_test.mean(), 3))

content_type share (limit: one type dominates):
content_type
keyword article       0.907
feedly article        0.070
comparison article    0.023
Name: proportion, dtype: float64

Grouped-split AUC (audited, from Section 3): 0.603
Naive-split AUC (from Section 3's honesty check): 0.752
Precision@50 vs. base rate: 0.54 vs. 0.517


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*


Ordered by the combination of observed decline rate and group size — largest,
most reliably-elevated-risk groups first:

1. **Stale Workhorse — refresh first.** 2,254 pages, 68.7% observed decline rate,
   real traffic already flowing. Staleness and visibility agree with the model here
   — the highest-confidence "go" group. `action: refresh_priority`
2. **General Decline Risk — review in batches.** 3,555 pages, 69.2% decline rate,
   flagged primarily by the model rather than a single obvious rule. Largest
   actionable group; work it in confidence-tiered batches. `action: refresh_review`
3. **Visible But Under-Clicked — cheap fix first.** 2,582 pages, 68.6% decline
   rate, high demand but low CTR. Try a title/meta rewrite before a full content
   rewrite. `action: rewrite_title_and_meta`
4. **Disengaging Reader — small but worth a look.** Only 221 pages, but the
   highest decline rate observed (69.2%) alongside a real engagement drop.
   `action: review_engagement_and_layout`
5. **Champion (protect) — explicit no-go list.** 2,647 pages, already winning
   (top position, high CTR). Do not touch — a "refresh" here risks breaking
   something that works. `action: protect_do_not_touch`
6. **No Real Demand — deprioritize.** 1,678 pages, 21.7% decline rate, no real
   search demand behind them. Not worth review time now. `action: monitor_only`
7. **Steady / No Flag — leave alone.** 17,062 pages, the bulk of the site,
   sitting near base rate with no flag raised. `action: monitor`

**Before anyone acts on a row:** confirm the reason code against the live page,
check confidence tier first (only the 10.6% high-confidence rows are close to
act-with-minimal-review), bring editorial/brand context the model never sees, and
treat this as an ordering aid for a human queue — never an auto-publish or
auto-rewrite trigger. Retrain roughly quarterly, or sooner if a wave of brand-new
clients arrives (the population this model is weakest on).

In [7]:
import os, numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.exists("flyrank-ml-internship"):
        get_ipython().system('git clone https://github.com/TheAlishbahWaheed/flyrank-ml-internship.git')
    os.chdir("flyrank-ml-internship")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

# Same leakage-safe feature set as Sections 3-4
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_scroll_data"] = df["scroll_rate"].notna().astype(int)

NUM_FEATS = ["search_volume","competition","cpc","word_count","char_count",
    "log_impressions_90d","log_clicks_90d","log_sessions_90d","log_ai_sessions_90d",
    "days_with_impressions","days_with_sessions","content_age_days","days_since_last_update",
    "ctr","avg_position","engagement_rate","scroll_rate","ai_traffic_pct",
    "has_keyword_data","has_word_count","has_scroll_data"]
CAT_FEATS = ["competition_level","content_type","main_intent","age_tier","freshness_tier",
    "word_count_tier","impression_tier","position_tier"]
for c in ["search_volume","competition","cpc","word_count","char_count","scroll_rate"]:
    df[c] = df[c].fillna(0)
for c in CAT_FEATS:
    df[c] = df[c].fillna("unknown").astype(str)

X = df[NUM_FEATS + CAT_FEATS]
y = df["is_declining_label"].values
groups = df["client_id"].values

def make_rf():
    pre = ColumnTransformer([("num", "passthrough", NUM_FEATS), ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATS)])
    return Pipeline([("pre", pre), ("clf", RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=RANDOM_STATE, n_jobs=-1))])

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(y_true)[order[:k]].mean())

# Audited number (matches Section 4 / w06): single grouped split
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(df, y, groups))
rf_single = make_rf().fit(X.iloc[train_idx], y[train_idx])
proba_test = rf_single.predict_proba(X.iloc[test_idx])[:, 1]
auc_single = roc_auc_score(y[test_idx], proba_test)
print(f"Reported (matches Section 4): grouped-split AUC {auc_single:.3f}")

# 5-fold client-grouped OOF — scores the FULL population for the queue
gkf = GroupKFold(n_splits=5)
oof_proba = np.zeros(len(df))
fold_aucs = []
for tr_idx, te_idx in gkf.split(X, y, groups):
    rf = make_rf().fit(X.iloc[tr_idx], y[tr_idx])
    p = rf.predict_proba(X.iloc[te_idx])[:, 1]
    oof_proba[te_idx] = p
    fold_aucs.append(roc_auc_score(y[te_idx], p))
df["oof_model_proba"] = oof_proba
oof_auc = roc_auc_score(y, oof_proba)
print(f"OOF (used for queue): AUC {oof_auc:.3f} | per-fold {[round(a,3) for a in fold_aucs]}")

# Reason codes
df["stale_and_visible"] = df["freshness_tier"].isin(["91-180", "181+"]) & df["impression_tier"].isin(["moderate", "good", "excellent"])
df["high_demand_low_ctr"] = (df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["ctr"] < 0.5)
df["low_engagement"] = (df["sessions_90d"] >= 30) & (((df["engagement_rate"] > 0) & (df["engagement_rate"] < 30)) | ((df["scroll_rate"] > 0) & (df["scroll_rate"] < 30)))
df["thin_but_visible"] = df["word_count_tier"].eq("<1000") & df["impression_tier"].isin(["moderate", "good", "excellent"])
df["already_winning"] = df["position_tier"].isin(["top_3", "page_1"]) & (df["ctr"] >= 0.5)
df["no_real_demand"] = (df["impression_tier"] == "low") & (df["has_keyword_data"] == 0)
df["model_decline_risk"] = df["oof_model_proba"] >= 0.65

def archetype(row):
    if row["already_winning"]: return "Champion (protect)"
    if row["thin_but_visible"] and row["model_decline_risk"]: return "Thin But Wanted"
    if row["stale_and_visible"] and row["model_decline_risk"]: return "Stale Workhorse"
    if row["high_demand_low_ctr"] and row["model_decline_risk"]: return "Visible But Under-Clicked"
    if row["low_engagement"] and row["model_decline_risk"]: return "Disengaging Reader"
    if row["model_decline_risk"]: return "General Decline Risk"
    if row["no_real_demand"]: return "No Real Demand"
    return "Steady / No Flag"
df["archetype"] = df.apply(archetype, axis=1)

ARCHETYPE_ACTION = {
    "Champion (protect)": "protect_do_not_touch", "Thin But Wanted": "expand_and_refresh",
    "Stale Workhorse": "refresh_priority", "Visible But Under-Clicked": "rewrite_title_and_meta",
    "Disengaging Reader": "review_engagement_and_layout", "General Decline Risk": "refresh_review",
    "No Real Demand": "monitor_only", "Steady / No Flag": "monitor",
}
df["action"] = df["archetype"].map(ARCHETYPE_ACTION)

def confidence(row):
    if row["model_decline_risk"] and row["impressions_90d"] >= 500 and row["sessions_90d"] >= 10: return "high"
    if row["oof_model_proba"] >= 0.5: return "medium"
    return "low"
df["confidence"] = df.apply(confidence, axis=1)

review = (df.groupby("archetype").agg(n=("content_id", "size"), decline_rate=("is_declining_label", "mean"),
          avg_oof_proba=("oof_model_proba", "mean")).sort_values("n", ascending=False))
print("\n", review.round(3))
print("\nAction counts:\n", df["action"].value_counts())
print("\nConfidence counts:\n", df["confidence"].value_counts())

Reported (matches Section 4): grouped-split AUC 0.603
OOF (used for queue): AUC 0.673 | per-fold [np.float64(0.654), np.float64(0.602), np.float64(0.735), np.float64(0.647), np.float64(0.662)]

                                n  decline_rate  avg_oof_proba
archetype                                                    
Steady / No Flag           17062         0.511          0.507
General Decline Risk        3555         0.692          0.700
Champion (protect)          2647         0.471          0.541
Visible But Under-Clicked   2582         0.686          0.696
Stale Workhorse             2254         0.687          0.702
No Real Demand              1678         0.217          0.295
Disengaging Reader           221         0.692          0.701
Thin But Wanted                1         1.000          0.698

Action counts:
 action
monitor                         17062
refresh_review                   3555
protect_do_not_touch             2647
rewrite_title_and_meta           2582
refresh_p

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*



Three figures and two tables, generated once by `w07_action_playbook.ipynb` and
committed to `work/figures/` so the deployed page and this notebook show the same
numbers:

- `precision_at_k.png` — baseline vs. LogReg vs. RandomForest, K=10..200 (Section 4)
- `archetype_counts.png` — page counts by archetype, 30,000 pages
- `archetype_decline_rate.png` — observed decline rate by archetype vs. base rate
- Table: precision@K (Section 4)
- Table: archetype summary — n, decline rate, avg score (Section 6)

![Archetype counts](https://github.com/TheAlishbahWaheed/flyrank-ml-internship/blob/main/work/figures/archetype_counts.png?raw=1)
*Most pages (56.9%) carry no flag at all — the queue is deliberately selective,
not a blanket alarm.*

![Decline rate by archetype](https://github.com/TheAlishbahWaheed/flyrank-ml-internship/blob/main/work/figures/archetype_decline_rate.png?raw=1)
*"Stale Workhorse," "Visible But Under-Clicked," and "General Decline Risk" all sit
~15-19 points above the 54.2% base rate; "No Real Demand" and "Champion" sit well
below it.*

In [8]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("work/figures", exist_ok=True)
os.makedirs("work/outputs", exist_ok=True)

# Table: precision@K for the deployed (OOF, whole-population) queue
precision_curve = {k: round(precision_at_k(y, oof_proba, k), 3) for k in [50, 100, 200, 500, 1000]}
print("Precision@K (deployed OOF queue):", precision_curve)
print("Base rate (whole dataset):", round(y.mean(), 3))

# Figure 1: archetype counts
fig, ax = plt.subplots(figsize=(8, 4.5))
order = df["archetype"].value_counts()
ax.barh(order.index[::-1], order.values[::-1], color="#426B69")
ax.set_xlabel("Number of content items"); ax.set_title("Content items by archetype")
plt.tight_layout(); plt.savefig("work/figures/archetype_counts.png", dpi=150); plt.close()

# Figure 2: decline rate by archetype vs base rate
fig, ax = plt.subplots(figsize=(8, 4.5))
rates = df.groupby("archetype")["is_declining_label"].mean().sort_values()
ax.barh(rates.index, rates.values, color="#8C6BB1")
ax.axvline(df["is_declining_label"].mean(), color="black", linestyle="--", linewidth=1, label="Overall base rate")
ax.set_xlabel("Observed decline rate"); ax.set_title("Decline rate by archetype vs. overall base rate"); ax.legend()
plt.tight_layout(); plt.savefig("work/figures/archetype_decline_rate.png", dpi=150); plt.close()

# Figure 3: precision@K of the ranked queue vs base rate
fig, ax = plt.subplots(figsize=(7, 4.5))
ks = [50, 100, 200, 500, 1000]
precs = [precision_at_k(y, oof_proba, k) for k in ks]
ax.plot([str(k) for k in ks], precs, marker="o", color="#4E79A7", label="Ranked-queue precision@K")
ax.axhline(y.mean(), color="black", linestyle="--", linewidth=1, label="Base rate")
ax.set_ylabel("Precision"); ax.set_xlabel("K"); ax.set_title("Ranked-queue precision@K vs. base rate"); ax.legend()
plt.tight_layout(); plt.savefig("work/figures/precision_at_k.png", dpi=150); plt.close()

print("\nWrote 3 figures to work/figures/ (should match the ones already in the repo — this just confirms reproducibility)")

Precision@K (deployed OOF queue): {50: 0.4, 100: 0.43, 200: 0.62, 500: 0.684, 1000: 0.714}
Base rate (whole dataset): 0.542

Wrote 3 figures to work/figures/ (should match the ones already in the repo — this just confirms reproducibility)


## 8. Demo outline — 5-minute Week-8 showcase (ML-12)

*Optional to present, required to have written. Timed for a 5-minute slot; the chart referenced is `work/figures/precision_at_k.png` (also embedded in the deployed paper's Results section).*

**1. Question — 30 sec**
Content teams manage a few thousand live pages and can't review all of them every week. Today the pick of what to review first is usually a hand-written rule ("anything untouched for 90 days") or a gut call. Can a model, trained only on observed search and content signals, rank pages by decline risk better than that rule — and by how much, once the comparison is made honest?

**2. Method — 1 min**
Built a ranking model (Logistic Regression + Random Forest) on a 30,000-page, 32-client anonymized FlyRank slice, using 29 observed features (demand, content, lifecycle, current-state rates) and an explicit-missingness flag rather than a silent `fillna(0)`. Compared it against a transparent "stale + visible" baseline. Validated with a **client-grouped** split (`GroupShuffleSplit`, zero client overlap between train and test) — the same pipeline re-run under a naive random split showed exactly why that matters.

**3. One chart — 1.5 min**
Show `precision_at_k.png`: precision@K for baseline vs. Logistic Regression vs. Random Forest on the identical grouped, held-out test clients. Walk the eye from K=10 to K=200 — the model leads at every depth, and the gap narrows as K widens, which is itself worth a sentence (most of the value shows up across hundreds of ranked rows, not a handful of top picks).

**4. One honest result — 1 min**
The headline lift: precision@10 of 0.90 (model) vs. 0.60 (baseline) on the honest, client-grouped split. But say the uncomfortable number out loud too — closing the client-leakage gap dropped ROC-AUC from 0.752 (naive split) to 0.603 (grouped split), a −0.149 drop purely from doing the validation honestly. That drop *is* the finding as much as the lift is.

**5. One recommendation — 1 min**
Ship the ranked, reason-coded queue as a decision-support tool, not an autopilot: start with "Stale Workhorse" (2,254 pages, 68.7% observed decline rate, staleness and visibility agree) as the highest-confidence refresh-first group, and route "Champion" pages to an explicit do-not-touch list. Retrain roughly quarterly, and re-check sooner if a wave of brand-new, never-seen clients arrives — exactly the population this model is weakest on.


## 9. Two shareable cuts (ML-12)

*Same work, retold for two audiences. Both are pasteable as-is.*

### Social post (methodology-focused)

> Spent this internship on a question every content team quietly guesses at: which of thousands of pages should get reviewed first? The easy answer is a staleness rule ("90 days untouched → flag it"). I tested whether a model trained on observed FlyRank search signals could beat that rule — and built in the honesty check most quick projects skip: a **client-grouped** train/test split, so no client's pages leak between the two.
>
> That check mattered. The same pipeline scored 0.752 ROC-AUC under a naive random split and 0.603 once client leakage was closed — a real, non-trivial drop, and the number I'm reporting. Even with that honesty tax, the model still roughly doubles the baseline's precision in the top ranks (precision@10: 0.90 vs 0.60).
>
> Full paper, reproducible notebooks, and the honest limitations section: [repo link]

### Employer-facing summary (3 sentences)

I built a machine-learning ranking model that scores which web pages are most likely declining in search performance, so a content team can prioritize a weekly review queue instead of relying on a blunt "90-days-stale" rule. It was trained and validated on a 30,000-page, 32-client anonymized slice of real FlyRank search-performance data, using a client-grouped train/test split so no client's pages could leak between training and evaluation. Under that honest split, the model roughly doubled the baseline rule's precision in the top-ranked pages (precision@10: 0.90 vs. 0.60) while also surfacing its own clearest weakness — performance on brand-new clients the model has never seen — and shipping that limitation directly in the report rather than hiding it.


## Self-check

Before you submit, confirm each line honestly:

- [ ✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅ ] No client names, URLs, or private queries anywhere
- [✅ ] My claims use careful words: observed, measured, directional, decision-support
- [ ✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ✅] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [✅ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
